# ⚡ Day 03a: Query Optimization — EXPLAIN, Query Plans, Performance Tips

---

## 🎯 What You'll Master Today
- Reading EXPLAIN output
- Query optimizer decisions
- Common anti-patterns that kill performance
- 12 optimization rules every developer should know

---

```
╔══════════════════════════════════════════════════════════════════╗
║            QUERY OPTIMIZER PIPELINE                              ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  SQL Query                                                       ║
║     ↓                                                            ║
║  Parser → Parse Tree                                             ║
║     ↓                                                            ║
║  Optimizer → generates candidate plans → picks cheapest          ║
║     ↓           (cost = I/O + CPU + memory)                      ║
║  Executor → runs the chosen plan                                 ║
║                                                                  ║
║  Access Methods:                                                 ║
║  • Full Table Scan    — reads every row                         ║
║  • Index Scan         — traverse index → lookup rows            ║
║  • Index-Only Scan    — all data in index (covering index!)     ║
║  • Bitmap Index Scan  — combine multiple indexes                ║
║                                                                  ║
║  Join Algorithms:                                                ║
║  • Nested Loop  — O(n×m). Good for small tables / indexed       ║
║  • Hash Join    — O(n+m). Build hash on smaller, probe larger   ║
║  • Sort-Merge   — O(n log n + m log m). Good for sorted data    ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Simulating EXPLAIN Output
# ============================================

import time

print("  === Reading EXPLAIN Output ===\n")

explain_examples = [
    {
        "query": "SELECT * FROM users WHERE id = 42",
        "plan": {
            "type": "Index Lookup",
            "table": "users",
            "key": "PRIMARY",
            "rows_examined": 1,
            "filtered": "100%",
            "extra": "",
        },
        "verdict": "✓ PERFECT — PK lookup, 1 row"
    },
    {
        "query": "SELECT * FROM users WHERE email = 'a@b.com'",
        "plan": {
            "type": "ALL (Full Scan)",
            "table": "users",
            "key": "None",
            "rows_examined": 1000000,
            "filtered": "10%",
            "extra": "Using where",
        },
        "verdict": "✗ BAD — Full scan! Add index on email"
    },
    {
        "query": "SELECT name FROM users WHERE status = 'active' ORDER BY created_at",
        "plan": {
            "type": "Index Scan",
            "table": "users",
            "key": "idx_status_created",
            "rows_examined": 5000,
            "filtered": "100%",
            "extra": "Using index condition",
        },
        "verdict": "✓ GOOD — Composite index covers WHERE + ORDER BY"
    },
]

for ex in explain_examples:
    print(f"  Query: {ex['query']}")
    plan = ex['plan']
    print(f"    type: {plan['type']}")
    print(f"    key: {plan['key']}")
    print(f"    rows: {plan['rows_examined']:,}")
    print(f"    filtered: {plan['filtered']}")
    print(f"    extra: {plan['extra']}")
    print(f"    → {ex['verdict']}\n")

In [ ]:
# ============================================
# 2. Join Algorithm Comparison
# ============================================

import random

# Generate data
n, m = 1000, 500
table_a = [{"id": i, "val": random.randint(1, 100)} for i in range(n)]
table_b = [{"id": random.randint(0, n-1), "data": f"item_{i}"} for i in range(m)]

# Nested Loop Join
start = time.perf_counter()
nl_result = []
comparisons = 0
for a in table_a:
    for b in table_b:
        comparisons += 1
        if a["id"] == b["id"]:
            nl_result.append((a["id"], a["val"], b["data"]))
nl_time = time.perf_counter() - start

# Hash Join
start = time.perf_counter()
hash_table = {}
for b in table_b:  # Build phase (smaller table)
    hash_table.setdefault(b["id"], []).append(b)
hj_result = []
hash_ops = len(table_b)
for a in table_a:  # Probe phase
    hash_ops += 1
    if a["id"] in hash_table:
        for b in hash_table[a["id"]]:
            hj_result.append((a["id"], a["val"], b["data"]))
hj_time = time.perf_counter() - start

# Sort-Merge Join
start = time.perf_counter()
sorted_a = sorted(table_a, key=lambda x: x["id"])
sorted_b = sorted(table_b, key=lambda x: x["id"])
sm_result = []
i = j = 0
while i < len(sorted_a) and j < len(sorted_b):
    if sorted_a[i]["id"] == sorted_b[j]["id"]:
        # Handle duplicates
        k = j
        while k < len(sorted_b) and sorted_b[k]["id"] == sorted_a[i]["id"]:
            sm_result.append((sorted_a[i]["id"], sorted_a[i]["val"], sorted_b[k]["data"]))
            k += 1
        i += 1
    elif sorted_a[i]["id"] < sorted_b[j]["id"]:
        i += 1
    else:
        j += 1
sm_time = time.perf_counter() - start

print(f"  Join Algorithm Benchmark ({n}×{m} tables):")
print(f"  ┌──────────────┬────────────────┬──────────┬─────────┐")
print(f"  │ Algorithm     │ Comparisons    │ Results  │ Time    │")
print(f"  ├──────────────┼────────────────┼──────────┼─────────┤")
print(f"  │ Nested Loop   │ {comparisons:>14,} │ {len(nl_result):>8} │ {nl_time:.4f}s │")
print(f"  │ Hash Join     │ {hash_ops:>14,} │ {len(hj_result):>8} │ {hj_time:.4f}s │")
print(f"  │ Sort-Merge    │      (sorted)  │ {len(sm_result):>8} │ {sm_time:.4f}s │")
print(f"  └──────────────┴────────────────┴──────────┴─────────┘")
print(f"\n  💡 Hash Join is fastest for equality joins. Optimizer picks automatically!")

In [ ]:
# ============================================
# 3. Anti-Patterns That Kill Performance
# ============================================

print("""
  ╔═══════════════════════════════════════════════════════════════════════╗
  ║              SQL ANTI-PATTERNS & FIXES                               ║
  ╠═══════════════════════════════════════════════════════════════════════╣
  ║                                                                       ║
  ║  1. Function on indexed column                                       ║
  ║     ✗ WHERE YEAR(created_at) = 2024                                  ║
  ║     ✓ WHERE created_at >= '2024-01-01' AND created_at < '2025-01-01'║
  ║     Why: function disables index usage (full scan!)                  ║
  ║                                                                       ║
  ║  2. Implicit type conversion                                         ║
  ║     ✗ WHERE phone = 1234567890  (phone is VARCHAR)                   ║
  ║     ✓ WHERE phone = '1234567890'                                     ║
  ║     Why: cast on every row → no index                                ║
  ║                                                                       ║
  ║  3. SELECT *                                                          ║
  ║     ✗ SELECT * FROM users WHERE id = 1                               ║
  ║     ✓ SELECT name, email FROM users WHERE id = 1                     ║
  ║     Why: fetches unnecessary columns, breaks covering index          ║
  ║                                                                       ║
  ║  4. OR that prevents index usage                                     ║
  ║     ✗ WHERE dept_id = 1 OR name = 'Alice'                           ║
  ║     ✓ Use UNION: (WHERE dept_id=1) UNION (WHERE name='Alice')       ║
  ║                                                                       ║
  ║  5. NOT IN with NULLs                                               ║
  ║     ✗ WHERE id NOT IN (SELECT id FROM other)  -- may contain NULL    ║
  ║     ✓ WHERE NOT EXISTS (SELECT 1 FROM other WHERE other.id = t.id)  ║
  ║                                                                       ║
  ║  6. N+1 query problem                                                ║
  ║     ✗ Loop: SELECT order FROM orders WHERE user_id = ?  (per user)   ║
  ║     ✓ SELECT * FROM orders WHERE user_id IN (1,2,3,...) (batch)      ║
  ║                                                                       ║
  ╚═══════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ============================================
# 4. 12 Optimization Rules
# ============================================

print("""
  ┌──────────────────────────────────────────────────────────────────┐
  │          12 SQL OPTIMIZATION RULES                               │
  ├──────────────────────────────────────────────────────────────────┤
  │                                                                  │
  │  INDEXING:                                                       │
  │  1. Index columns in WHERE, JOIN, ORDER BY                      │
  │  2. Use composite indexes (most selective column first)         │
  │  3. Covering indexes eliminate table lookups                     │
  │  4. Don't over-index: each index slows writes                   │
  │                                                                  │
  │  QUERY DESIGN:                                                   │
  │  5. Avoid SELECT * — specify needed columns                     │
  │  6. Never put functions on indexed columns in WHERE             │
  │  7. Use EXISTS instead of IN for large subqueries               │
  │  8. Batch operations (avoid N+1 queries)                         │
  │                                                                  │
  │  SCHEMA:                                                         │
  │  9. Normalize for writes, denormalize for reads                 │
  │  10. Use appropriate data types (INT vs VARCHAR for IDs)        │
  │  11. Partition large tables (by date, range, or hash)            │
  │  12. Archive/delete old data (keep working set small)           │
  │                                                                  │
  │  EXPLAIN workflow: EXPLAIN → identify bottleneck →              │
  │  add index OR rewrite query → EXPLAIN again → verify            │
  │                                                                  │
  └──────────────────────────────────────────────────────────────────┘
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | What does EXPLAIN show? | Access type, key used, rows examined, extra info (filesort, temp table) |
| 2 | Nested Loop vs Hash Join? | NL: O(n×m) good with index. Hash: O(n+m) best for equality joins |
| 3 | Why is SELECT * bad? | Fetches unnecessary data, breaks covering index, wastes I/O |
| 4 | How to optimize slow query? | EXPLAIN → check type/key/rows → add index → rewrite → EXPLAIN again |
| 5 | What is N+1 problem? | 1 query for list + N queries per item. Fix: batch with IN or JOIN |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • EXPLAIN is your #1 debugging tool for slow queries   │
## │  • Hash Join > Nested Loop for large equality joins     │
## │  • Never apply functions on indexed columns in WHERE    │
## │  • Covering index = index-only scan = fastest           │
## │  • Follow the 12 rules: index, query design, schema    │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **NoSQL Databases** — Document, Key-Value, Column, Graph